# Production Benchmark Runner

Each code cell builds one part of the experiment and prints a short audit summary. Choose a YAML config, then run cells top to bottom.

### Environment

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

os.environ["MNE_DONTWRITE_HOME"] = "true"
os.environ["NUMBA_DISABLE_JIT"] = "1"
os.environ["MPLCONFIGDIR"] = "/tmp/neurosned-matplotlib"

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.config import load_experiment_config, resolve_path
from benchmarks.utils import set_seed

CONFIG_PATH = PROJECT_ROOT / "benchmarks/configs/segmentation/unet_deeper_demo.yaml"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print(f"Device: {device}")
print(f"Torch: {torch.__version__}")

### Load Config

In [ ]:
config = load_experiment_config(CONFIG_PATH)
set_seed(config.seed)

print(f"Experiment: {config.name}")
print(f"Task: {config.task}")
print(f"Seed: {config.seed}")
print(f"Model: {config.model.module_name}.{config.model.class_name}")
print(f"Trainer: {config.trainer.module_name}.{config.trainer.class_name}")
print(f"Optimizer: {config.optimizer.module_name}.{config.optimizer.class_name}")

### Load Datasets

In [ ]:
train_dataset, valid_dataset = config.build_datasets(PROJECT_ROOT)
data_paths = config.data_paths(PROJECT_ROOT)

print(f"Train path: {data_paths['train'].relative_to(PROJECT_ROOT)}")
print(f"Valid path: {data_paths['valid'].relative_to(PROJECT_ROOT)}")
if data_paths["test"] is not None:
    print(f"Test path: {data_paths['test'].relative_to(PROJECT_ROOT)}")
print(f"Train windows: {len(train_dataset):,}")
print(f"Valid windows: {len(valid_dataset):,}")

### Inspect Targets

In [ ]:
meta_information = train_dataset.get_metadata()
meta_information_valid = valid_dataset.get_metadata()

def target_summary(name, metadata):
    target = metadata["target"]
    return {
        "split": name,
        "rows": len(metadata),
        "target_mean": float(target.mean()),
        "target_std": float(target.std()),
        "target_min": float(target.min()),
        "target_max": float(target.max()),
    }

summary = pd.DataFrame([
    target_summary("train", meta_information),
    target_summary("valid", meta_information_valid),
])
default_rmse_train = meta_information["target"].std()
default_rmse_valid = meta_information_valid["target"].std()

print(summary.to_string(index=False))
print(f"Default RMSE denominator: train={default_rmse_train:.4f}, valid={default_rmse_valid:.4f}")

### Build Model

In [ ]:
model = config.model.build().to(device)

input_checkpoint_path = resolve_path(config.trainer.checkpoint.input, PROJECT_ROOT)
output_checkpoint_path = resolve_path(config.trainer.checkpoint.output, PROJECT_ROOT)
output_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

checkpoint_loaded = False
if input_checkpoint_path is not None and input_checkpoint_path.exists():
    model.load_state_dict(torch.load(input_checkpoint_path, map_location=device))
    checkpoint_loaded = True
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
mb_total = total_params * 4 / 1024**2

print(f"Model class: {config.model.class_name}")
print(f"Parameters: {total_params:,} ({mb_total:.2f} MB float32)")
print(f"Input checkpoint: {input_checkpoint_path if input_checkpoint_path is not None else 'None'}")
print(f"Checkpoint loaded: {checkpoint_loaded}")
print(f"Output checkpoint: {output_checkpoint_path}")

### Build Dataset Wrappers

In [ ]:
channels_list = np.arange(model.n_chans)

train_dataset_for_loader = train_dataset
if config.data.train_dataset is not None:
    train_dataset_for_loader = config.data.train_dataset.build(train_dataset, use_channels=channels_list)

valid_dataset_for_loader = valid_dataset
if config.data.valid_dataset is not None:
    valid_dataset_for_loader = config.data.valid_dataset.build(valid_dataset)

train_wrapper = config.data.train_dataset.class_name if config.data.train_dataset is not None else "None"
valid_wrapper = config.data.valid_dataset.class_name if config.data.valid_dataset is not None else "None"

print(f"Channels: {len(channels_list)}")
print(f"Train wrapper: {train_wrapper} | rows={len(train_dataset_for_loader):,}")
print(f"Valid wrapper: {valid_wrapper} | rows={len(valid_dataset_for_loader):,}")

### Build Loaders

In [ ]:
train_loader = DataLoader(train_dataset_for_loader, **config.loaders.train.to_kwargs())
valid_loader = DataLoader(valid_dataset_for_loader, **config.loaders.valid.to_kwargs())

print(f"Train loader: batches={len(train_loader):,}, batch_size={config.loaders.train.batch_size}, workers={config.loaders.train.num_workers}, shuffle={config.loaders.train.shuffle}")
print(f"Valid loader: batches={len(valid_loader):,}, batch_size={config.loaders.valid.batch_size}, workers={config.loaders.valid.num_workers}, shuffle={config.loaders.valid.shuffle}")

### Build Trainer

In [ ]:
from benchmarks.training import ReloadBestOnPlateau

optimizer_cls = config.optimizer.load_class()
optimizer = optimizer_cls(model.parameters(), **config.optimizer.params)
optimizer_kwargs = {key: value for key, value in config.optimizer.params.items() if key != "lr"}

plateau_scheduler = None
if config.trainer.plateau.enabled:
    plateau_scheduler = ReloadBestOnPlateau(
        optimizer_factory=optimizer_cls,
        lr=config.optimizer.params["lr"],
        factor=config.trainer.plateau.factor,
        optimizer_kwargs=optimizer_kwargs,
        fallback_checkpoint_path=input_checkpoint_path,
    )

trainer_cls = config.trainer.load_class()
trainer = trainer_cls(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    device=device,
    n_epochs=config.trainer.n_epochs,
    checkpoint_path=output_checkpoint_path,
    monitor=config.trainer.monitor,
    minimize=config.trainer.minimize,
    early_stopping_patience=config.trainer.early_stopping_patience,
    plateau_scheduler=plateau_scheduler,
    print_batch_stats=config.trainer.print_batch_stats,
    channels_list=channels_list,
    default_rmse=default_rmse_valid,
    **config.trainer.params,
)

print(f"Trainer: {type(trainer).__name__}")
print(f"Epochs: {trainer.n_epochs}")
print(f"Monitor: {trainer.monitor} | minimize={trainer.minimize}")
print(f"Patience: {trainer.early_stopping_patience}")
print(f"Optimizer: {config.optimizer.class_name} | params={config.optimizer.params}")
print(f"Plateau: enabled={config.trainer.plateau.enabled}, factor={config.trainer.plateau.factor}")

### Initial Validation

In [ ]:
initial_valid_metrics = None
if input_checkpoint_path is not None and input_checkpoint_path.exists():
    initial_valid_metrics = trainer.run_valid_epoch(0)
    trainer.best_metric = initial_valid_metrics[config.trainer.monitor]
    trainer.best_epoch = 0
    print(f"Initial {config.trainer.monitor}: {trainer.best_metric:.6f}")
else:
    print("Skipped: no input checkpoint found.")

### Run Training

In [ ]:
history = trainer.run()
best_metric = trainer.best_metric
best_epoch = trainer.best_epoch

print(f"Best {config.trainer.monitor}: {best_metric:.6f}")
print(f"Best epoch: {best_epoch}")
print(f"Saved checkpoint: {output_checkpoint_path}")

### Inspect History

In [ ]:
history_df = pd.DataFrame(history)

if history_df.empty:
    print("History is empty.")
else:
    print(history_df.tail().to_string(index=False))
    valid_col = f"valid_{config.trainer.monitor}"
    if valid_col in history_df:
        best_row = history_df.loc[history_df[valid_col].idxmin() if config.trainer.minimize else history_df[valid_col].idxmax()]
        print(f"Best row by {valid_col}: epoch={int(best_row['epoch'])}, value={best_row[valid_col]:.6f}")

### Continue Experiments

Copy a YAML config, adjust the controlled section, then point `CONFIG_PATH` to the new file.